# Data Preprocessing Sanity check
## Choose dataset, preprocessing method, and review results

This notebook provides a unified interface to:
1. Load data
2. Choose a preprocessing method (basic, **Llama**, or **Gemini API**)
3. **Test with a small sample first** (NUM_SAMPLES = 5)
4. Review and compare results
5. Run on full dataset when satisfied

## Configuration

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import pprint
import google.generativeai as genai
from dotenv import load_dotenv

# Add project root to path
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)  
sys.path.append(project_root)
sys.path.append(os.path.join(project_root, "src"))  # Add src to path


try:
    from mosaic.preprocessing.translation_utils import (
        list_preprocessed_datasets,
        TruncationDiagnostic,
        compare_word_counts,
        compare_source_and_translated,
        show_preprocessing_stats,
        resolve_data_path
    )
except ImportError as e:
    print(f"Error loading modules: {e}")


print("Preprocessing module loaded successfully")

Preprocessing module loaded successfully


## Check inventory of available (preprocessed) datasets

In [3]:
# Auto-detect all available preprocessed data
inventory = list_preprocessed_datasets()

if "error" in inventory:
    print(f"Error: {inventory['error']}")
else:
    print(f"Found {len(inventory)} datasets with preprocessed files:\n")
    for name, data in inventory.items():
        print(f"{name} ({data['total_reports']} reports)")
        for f in data['files']:
            print(f"   - {f['filename']} ({f['size_mb']} MB) [{f['method'].upper()}]")
        print()

Found 5 datasets with preprocessed files:

MPE_cleaned_llama_5_test.csv (5 reports)
   - MPE_cleaned_llama_5_test.csv (0.02 MB) [LLAMA]

dreamachine_DL (98 reports)
   - dreamachine_DL_preprocessed.csv (0.03 MB) [BASIC]

dreamachine_HS (334 reports)
   - dreamachine_HS_preprocessed.csv (0.09 MB) [BASIC]

ganzfeld_GREEN (34 reports)
   - ganzfeld_GREEN_cleaned_API.csv (0.04 MB) [GEMINI]
   - ganzfeld_GREEN_cleaned_llama.csv (0.05 MB) [LLAMA]

ganzfeld_RED (34 reports)
   - ganzfeld_RED_cleaned_llama.csv (0.06 MB) [LLAMA]



## Configutation: select Dataset and Preprocessing Method



In [7]:
# ========================================================
# CONFIGURATION
# ========================================================

# Paste the filename you want to check (e.g. 'MPE_cleaned_API.csv')
TARGET_FILE = 'ganzfeld_GREEN_cleaned_llama.csv'

# Column names (usually these defaults work)
SOURCE_COL = 'reflection_answer'
TARGET_COL = 'cleaned_reflection'

# ========================================================

# FIX: Add 'DATA/' to the path string
full_path = resolve_data_path(f"DATA/preprocessed/{TARGET_FILE}")

if full_path.exists():
    print(f"Selected file: {TARGET_FILE}")
    print(f"   Path: {full_path}")
else:
    print(f"File not found: {TARGET_FILE}")
    print(f"   Checked path: {full_path}")
    print("Check the filename from the list above.")

Selected file: ganzfeld_GREEN_cleaned_llama.csv
   Path: /Users/rb666/Projects/MOSAIC/DATA/preprocessed/ganzfeld_GREEN_cleaned_llama.csv


In [9]:
#Checks row counts and looks for an associated error log.
show_preprocessing_stats(str(full_path))


PREPROCESSING STATISTICS

Loading preprocessed data from: /Users/rb666/Projects/MOSAIC/DATA/preprocessed/ganzfeld_GREEN_cleaned_llama.csv
Loaded 34 rows
Reports (rows):        34
Note: No 'sentences' column found (run basic_preprocess first)

No error log found




In [11]:
#check truncation: Calculates retention rates and checks for cut-off text.
diagnostic = TruncationDiagnostic(
    str(full_path), 
    source_col=SOURCE_COL, 
    target_col=TARGET_COL
)

diagnostic.run_full_diagnostic()


DATA TRUNCATION DIAGNOSTIC REPORT

File: /Users/rb666/Projects/MOSAIC/DATA/preprocessed/ganzfeld_GREEN_cleaned_llama.csv
Rows: 34
Columns: ['reflection_answer', 'cleaned_reflection']

1. ERROR MARKERS (Processed with Errors)
----------------------------------------------------------------------------------------------------
✅ No error markers detected.

2. TOKEN LIMIT ANALYSIS (LLM max_tokens setting)
----------------------------------------------------------------------------------------------------
LLM max_tokens setting: 8192
Estimated max chars:    32768

Texts at risk (>90% of limit):  0 (0.0%)
Texts exceeding limit:          0
Max source text size:           2829 chars
Estimated max source tokens:    707 tokens

✅ Token limits seem adequate

3. DATA RETENTION ANALYSIS
----------------------------------------------------------------------------------------------------
Overall word retention rate:  96.0%
Total words lost:            187

Distribution of retention rates:
  Excellen

In [12]:
#visual comaparison of translation vs original
compare_source_and_translated(
    diagnostic.df, 
    source_col=SOURCE_COL, 
    target_col=TARGET_COL, 
    num_samples=3,
    anonymise=False  # Set to True to mask names/PII in output
)


COMPARISON: reflection_answer → cleaned_reflection (3 examples)

[Row 4]
SOURCE (1932 chars):
La première expérience (lumière blanche), je trouvais ça très apaisant, comme j’ai dit avec la lumière blanche… En fait, toutes les questions étaient très pertinentes dans le questionnaire tout à l’heure. Je me sentais dans une paix intérieure, comme dans un rêve ou dans l’au-delà. C’était très… J’a
...

TARGET (1614 chars):
The first experience (white light), I found it very soothing, as I said with the white light... In fact, all the questions were very relevant in the questionnaire just now. I felt at peace, like in a dream or beyond. It was very... I had many images, I said to myself, wow, it's hallucinations, I saw
...
----------------------------------------------------------------------------------------------------

[Row 26]
SOURCE (513 chars):
J’ai bien apprécié l’expérience. C’était long. Ça paraît très très long. Et difficile de rester les yeux ouverts. Je croyais que j’allais avoi